# Check your bank statement with ForgeLens

ForgeLens is a final-year project at SRM that detects forged bank statements. To measure how often it **wrongly** flags a genuine statement, we need real statements, but we don't want yours.

This notebook runs ForgeLens **in your own Google account**. Your statement is uploaded only to this temporary Colab machine, which is deleted when you close it. At the end you download one small file with **scores only**: no name, account number, date, amount or transaction. You can open it and check before sending it to us.

**Steps:** run each cell in order (the ▶ button, or Shift+Enter). It takes about 2–5 minutes.

*Faster:* Runtime → Change runtime type → T4 GPU (optional).

In [ ]:
# 1. Get ForgeLens (public code) and the two libraries it needs
!git clone -q --depth 1 -b review/engineering-review https://github.com/wahiddd-eng/forgelens
%cd forgelens
!pip -q install pymupdf easyocr
import sys; sys.path.insert(0, 'src'); sys.path.insert(0, 'scripts')
print('Ready')

In [ ]:
# 2. Upload your statement (the PDF exactly as the bank gave it: don't edit or re-save it)
from google.colab import files
uploaded = files.upload()
pdf = next(iter(uploaded))
print('Uploaded one file')

In [ ]:
# 3. Tell us the bank and how you got the statement
bank = input('Bank name (e.g. HDFC Bank): ')
how = input('How did you get it (net banking / email / app / branch)? ')

In [ ]:
# 4. Run ForgeLens (asks for the password if the PDF has one; it is never stored)
import getpass, json
from pathlib import Path
from forgelens.reader import PasswordRequired
from self_test import run
password = None
for attempt in range(3):
    try:
        result = run(Path(pdf), bank, how, password)
        break
    except PasswordRequired:
        password = getpass.getpass('This statement is password-protected. Password: ')
out = Path(f"/content/forgelens_result_{result['id']}.json")
out.write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))

In [ ]:
# 5. Download the result (scores only) and send it to the ForgeLens team
files.download(str(out))

In [ ]:
# 6. Delete your statement from this machine
import os
os.remove(pdf); password = None
print('Deleted. To remove everything: Runtime → Disconnect and delete runtime.')